## 0 · Setup & Ambiente

In [1]:
!pip install --upgrade ipykernel jupyter-core jupyter_http_over_ws
!jupyter serverextension enable --py jupyter_http_over_ws

INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 1.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 16.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 35.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: jupyter-client
    Found existing installation: jupyter_client 7.4.9
    Uninstalling jupyter_client-7.4.9:
      Successfully uninstalled jupyter_client-7.4.9
  A

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: builder console dejavu events execute kernel
kernelgateway kernelspec lab labextension labhub migrate nbclassic nbconvert
notebook run server troubleshoot trust

Jupyter command `jupyter-serverextension` not found.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import platform
import subprocess

print('Python:', sys.version.split()[0])
print('Plataforma:', platform.platform())
print('COLAB_GPU:', os.environ.get('COLAB_GPU', 'não definido'))
print('Diretório /content existe?:', os.path.exists('/content'))

try:
    smi = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False)
    if smi.returncode == 0 and smi.stdout.strip():
        print('nvidia-smi:', smi.stdout.strip())
    else:
        print('nvidia-smi: indisponível neste kernel')
except FileNotFoundError:
    print('nvidia-smi: comando não encontrado neste kernel')

try:
    import torch
    print('torch:', torch.__version__)
    print('torch.version.cuda:', torch.version.cuda)
    print('torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU ativa:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Falha ao verificar torch/cuda:', repr(e))

In [ ]:
import os, sys

REPO_PATH = '/content/classificador-fake-br'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone https://github.com/lisearantes/portuguese-fake-news-detection.git {REPO_PATH}')
else:
    os.system(f'git -C {REPO_PATH} pull --quiet')

sys.path.insert(0, REPO_PATH)

In [ ]:
import os
import sys
import subprocess

# Instala no MESMO ambiente Python do kernel ativo
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', f'{REPO_PATH}/requirements.txt',
        'transformers', 'datasets', 'accelerate', 'evaluate'
    ],
    check=True,
 )
subprocess.run(
    [sys.executable, '-m', 'spacy', 'download', 'pt_core_news_sm'],
    check=True,
 )

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Dependências instaladas no kernel ativo (incluindo ecossistema Hugging Face).')

In [ ]:
import matplotlib.font_manager as fm
import pandas as pd

os.system('wget -q -O Arial.ttf https://github.com/matomo-org/travis-scripts/raw/master/fonts/Arial.ttf')
fm.fontManager.addfont('Arial.ttf')

from src.visualization import configurar_fonte_arial
configurar_fonte_arial()

pd.set_option('display.max_colwidth', 140)
print('Setup concluído.')

## 1 · Carregamento do Corpus

In [ ]:
CSV_PATH  = '/content/drive/MyDrive/Fake.br-Corpus/fake.br-full_texts.csv'
FULL_PATH = '/content/drive/MyDrive/Fake.br-Corpus/full_texts/'

if not os.path.exists(FULL_PATH):
    raise FileNotFoundError(f'Corpus não encontrado em: {FULL_PATH}')

df = pd.read_csv(CSV_PATH)
print(f'Dimensões: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print(f'Tipos:\n{df.dtypes}')

## 2 · Análise Exploratória (EDA)

In [ ]:
from src.visualization import (
    plot_preenchimento,
    plot_distribuicao_classes,
    plot_distribuicao_categorias,
)

plot_preenchimento(df)
plot_distribuicao_classes(df)
plot_distribuicao_categorias(df)

## 3 · Pré-processamento de Texto

In [ ]:
from src.preprocessing import text_cleaning, wrap_texto

df['clean_text'] = df['texto_completo'].apply(text_cleaning)

In [ ]:
texto_original = df['texto_completo'].dropna().astype(str).iloc[0]
texto_limpo    = text_cleaning(texto_original)
n_orig  = len(texto_original.split())
n_limpo = len(texto_limpo.split())
compactacao = (n_orig - n_limpo) / n_orig * 100

print(f'Original ({n_orig} palavras):\n{wrap_texto(texto_original)}\n')
print(f'Limpo ({n_limpo} palavras | compactação: {compactacao:.1f}%):\n{wrap_texto(texto_limpo)}')

In [ ]:
from src.visualization import plot_boxplot_tamanho
plot_boxplot_tamanho(df)

## 4 · Vetorização TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
tfidf_terms   = vectorizer.get_feature_names_out()

print(f'Treino:    {X_train_tfidf.shape[0]} docs x {X_train_tfidf.shape[1]} termos')
print(f'Teste:     {X_test_tfidf.shape[0]} docs x {X_test_tfidf.shape[1]} termos')
print(f'Densidade: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.6f}')

In [ ]:
from src.visualization import plot_esparsidade_tfidf, plot_heatmap_tfidf

plot_esparsidade_tfidf(X_train_tfidf)
plot_heatmap_tfidf(X_train_tfidf, tfidf_terms)

## 5 · Modelagem e Avaliação

In [ ]:
from src.evaluation import treinar_svc, treinar_lr, relatorio_completo

modelo_svc, pred_svc = treinar_svc(X_train_tfidf, y_train, X_test_tfidf)
modelo_lr,  pred_lr  = treinar_lr(X_train_tfidf,  y_train, X_test_tfidf)



relatorio_completo(y_test, pred_svc, 'Linear SVC')
relatorio_completo(y_test, pred_lr,  'Logistic Regression')

In [ ]:
from src.evaluation import tabela_resultados

df_resultados = tabela_resultados(
    y_test,
    [(pred_svc, 'Linear SVC'), (pred_lr, 'Logistic Regression')],
)
print(df_resultados.to_string(index=False))

In [ ]:
from src.visualization import plot_confusion_matrix

plot_confusion_matrix(y_test, pred_svc)

## 6 · Fine-Tuning do BERTimbau

Nesta etapa, usamos o modelo BERTimbau para adaptar uma representação contextual em português ao problema de classificação entre notícias reais e falsas. A lógica do treinamento foi modularizada em [src/bert_finetuning.py](src/bert_finetuning.py), e o notebook mantém apenas a chamada do pipeline principal para manter o fluxo claro e reuso fácil.

In [ ]:
!pip install evaluate

In [ ]:
import importlib
import src.bert_finetuning

# Força o recarregamento do arquivo atualizado
importlib.reload(src.bert_finetuning)

from src.bert_finetuning import finetune_bertimbau
# 3. Chamada da função 
result = finetune_bertimbau(
    df,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    output_dir=os.path.join(REPO_PATH if os.path.exists(REPO_PATH) else os.getcwd(), 'results', 'bertimbau_fake_br')
)

In [ ]:
trainer = result['trainer']

print('Fine-tuning executado com sucesso via módulo src.bert_finetuning!')


In [ ]:
from src.visualization import plot_impacto_vetorizacao_transformer
os.makedirs('results/figures', exist_ok=True)

fig_transformer, transformer_visualization = plot_impacto_vetorizacao_transformer(
    model=result['model'],
    tokenizer=result['tokenizer'],
    texts=result['X_test'],
    labels=result['y_test'],
    output_path='results/figures/impacto_vetorizacao_transformer.png',
    max_samples=500,
    batch_size=16,
    max_length=512,
)

print('Visualizacao do impacto da vetorizacao do BERTimbau gerada com sucesso.')

Carrega modelo

In [ ]:
import os
import inspect
import numpy as np
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# 1. Ajuste dos caminhos
CAMINHO_BASE = os.path.join(REPO_PATH if os.path.exists(REPO_PATH) else os.getcwd(), "results", "bertimbau_fake_br")
CAMINHO_FINAL = os.path.join(CAMINHO_BASE, "modelo_final")
CAMINHO_MODELO = CAMINHO_FINAL if os.path.exists(CAMINHO_FINAL) else CAMINHO_BASE

# 2. Carregar Tokenizer e Modelo Salvo
tokenizer = AutoTokenizer.from_pretrained(CAMINHO_MODELO)


model = AutoModelForSequenceClassification.from_pretrained(CAMINHO_MODELO)

# 3. Métricas de Avaliação
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {"f1": f1["f1"], "accuracy": acc["accuracy"]}

# 4. Argumentos mínimos apenas para avaliação
eval_args = TrainingArguments(
    output_dir="./temp_eval_dir",
    per_device_eval_batch_size=16,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# 5. Instanciar o Trainer
trainer_kwargs = {
    "model": model,
    "args": eval_args,
    "eval_dataset": tokenized_datasets["test"] if "tokenized_datasets" in globals() else None,
    "data_collator": DataCollatorWithPadding(tokenizer=tokenizer),
    "compute_metrics": compute_metrics,
}

# Compatibilidade de versões da biblioteca
trainer_signature = inspect.signature(Trainer.__init__).parameters
if "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer instanciado com sucesso!")

## 7 · Avaliação do Modelo BERT

Nesta etapa, avaliamos o desempenho do modelo fine-tuned em todo o conjunto de teste. Apresentamos as seguintes métricas:
- **Accuracy**: Proporção geral de predições corretas
- **F1-Score**: Média harmônica entre Precision e Recall
- **Precision**: Proporção de predições positivas que foram corretas
- **Recall**: Proporção de exemplos positivos corretamente identificados
- **Matriz de Confusão**: Visualização dos verdadeiros/falsos positivos e negativos

In [75]:
import evaluate
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print("INICIANDO AVALIAÇÃO DO MODELO BERT FINE-TUNED")
print("=" * 70)

INICIANDO AVALIAÇÃO DO MODELO BERT FINE-TUNED


### 7.1 · Validação do Dataset

In [76]:
try:
    # Validar presença do dataset de teste
    if trainer.eval_dataset is None:
        raise ValueError("Dataset de teste não encontrado no Trainer. Verifique a inicialização.")
    
    print(f"✓ Dataset de teste carregado: {len(trainer.eval_dataset)} exemplos")
    print("✓ Prosseguindo com avaliação...")
    
except Exception as e:
    print(f"❌ ERRO: {type(e).__name__}")
    print(f"Detalhes: {str(e)}")
    print("\nDicas de resolução:")
    print("  • Verifique se o modelo foi carregado corretamente")
    print("  • Confirme se trainer.eval_dataset está disponível")
    raise

✓ Dataset de teste carregado: 1440 exemplos
✓ Prosseguindo com avaliação...


### 7.2 · Definição de Métricas e Função de Cálculo

In [77]:
# Definir função de cálculo de métricas com average="weighted"
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    # Calcular cada métrica com average="weighted"
    acc = evaluate.load("accuracy").compute(predictions=preds, references=labels)
    f1 = evaluate.load("f1").compute(predictions=preds, references=labels, average="weighted")
    prec = evaluate.load("precision").compute(predictions=preds, references=labels, average="weighted")
    rec = evaluate.load("recall").compute(predictions=preds, references=labels, average="weighted")
    
    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
        "precision": prec["precision"],
        "recall": rec["recall"],
    }

# Atualizar compute_metrics no Trainer
trainer.compute_metrics = compute_metrics

print("✓ Função de cálculo de métricas configurada com average='weighted'")

✓ Função de cálculo de métricas configurada com average='weighted'


### 7.3 · Execução da Avaliação

In [78]:
print("\nExecutando avaliação no conjunto de teste...")

# Executar avaliação
bert_eval = trainer.evaluate()

# Extrair valores com formatação
bert_acc = bert_eval.get("eval_accuracy", 0.0)
bert_prec = bert_eval.get("eval_precision", 0.0)
bert_rec = bert_eval.get("eval_recall", 0.0)
bert_f1 = bert_eval.get("eval_f1", 0.0)

print("✓ Avaliação concluída com sucesso!")

# Exibir resultados tabulares
print("\n" + "=" * 70)
print("MÉTRICAS DE DESEMPENHO - MODELO BERT")
print("=" * 70)
print(f"{'Acurácia':<30} {bert_acc:>10.4f}")
print(f"{'Precisão (weighted)':<30} {bert_prec:>10.4f}")
print(f"{'Recall (weighted)':<30} {bert_rec:>10.4f}")
print(f"{'F1-Score (weighted)':<30} {bert_f1:>10.4f}")


Executando avaliação no conjunto de teste...


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
No log,0.122733,0,0.973611,0.973609,0.973741,0.973611


✓ Avaliação concluída com sucesso!

MÉTRICAS DE DESEMPENHO - MODELO BERT
Acurácia                           0.9736
Precisão (weighted)                0.9737
Recall (weighted)                  0.9736
F1-Score (weighted)                0.9736


### 7.4 · Análise de Confiança e Predições Brutas

In [ ]:
print("\nExtraindo predições brutas...")

# Obter predições brutas
raw_preds = trainer.predict(trainer.eval_dataset)
y_true = raw_preds.label_ids
logits = raw_preds.predictions
y_pred = np.argmax(logits, axis=-1)

# Calcular confiança
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
y_pred_probs = np.max(probs, axis=-1)

print(f"✓ {len(y_pred)} predições extraídas")
print(f"  Confiança média: {y_pred_probs.mean():.4f}")

### 7.5 · Matriz de Confusão Detalhada

In [ ]:
print("\nGerando matriz de confusão...")

# Calcular matriz de confusão
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 70)
print("MATRIZ DE CONFUSÃO DETALHADA")
print("=" * 70)
print(f"{'Verdadeiros Negativos (TN)':<30} {tn:>10} (notícias reais corretas)")
print(f"{'Falsos Positivos (FP)':<30} {fp:>10} (reais classificadas como fake)")
print(f"{'Falsos Negativos (FN)':<30} {fn:>10} (fake classificadas como reais)")
print(f"{'Verdadeiros Positivos (TP)':<30} {tp:>10} (notícias fake corretas)")
print(f"{'TOTAL':<30} {tn+fp+fn+tp:>10}")

### 7.6 · Relatório de Classificação por Classe

In [ ]:
print("\n" + "=" * 70)
print("RELATÓRIO DE CLASSIFICAÇÃO POR CLASSE")
print("=" * 70)
print(classification_report(
    y_true, y_pred, 
    target_names=["Real (0)", "Fake (1)"],
    digits=4
))

### 7.7 · Visualização da Matriz de Confusão

In [ ]:
# Visualizar matriz de confusão
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=["Real (0)", "Fake (1)"],
    yticklabels=["Real (0)", "Fake (1)"],
    cbar_kws={'label': 'Frequência'},
    annot_kws={'fontsize': 12, 'weight': 'bold'}
)
plt.title('Matriz de Confusão - Modelo BERT Fine-Tuned', fontsize=14, fontweight='bold')
plt.ylabel('Verdadeiro', fontsize=12)
plt.xlabel('Predito', fontsize=12)
plt.tight_layout()

# Criar diretório se não existir
import os
os.makedirs('./results/figures', exist_ok=True)

plt.savefig('./results/figures/confusion_matrix_bert.png', dpi=300, bbox_inches='tight')
print("\n✓ Matriz de confusão salva em: ./results/figures/confusion_matrix_bert.png")
plt.show()

### 7.8 · Comparação Geral de Todos os Modelos

In [ ]:
# Criar DataFrame com resultados do BERT
linha_bert = pd.DataFrame([{
    "Modelo": "BERTimbau (fine-tuned)",
    "Precisão": f"{bert_prec:.4f}",
    "Recall": f"{bert_rec:.4f}",
    "F1-Score": f"{bert_f1:.4f}",
    "Acurácia": f"{bert_acc:.4f}",
}])

# Concatenar com resultados anteriores
if "df_resultados" in globals() and df_resultados is not None:
    # Formatar os modelos anteriores também
    df_resultados_formatado = df_resultados.copy()
    for col in ["Precisão", "Recall", "F1-Score", "Acurácia"]:
        if col in df_resultados_formatado.columns:
            df_resultados_formatado[col] = df_resultados_formatado[col].apply(
                lambda x: f"{float(x):.4f}" if x is not None else "N/A"
            )
    df_resultados_geral = pd.concat([df_resultados_formatado, linha_bert], ignore_index=True)
else:
    df_resultados_geral = linha_bert

print("\n" + "=" * 70)
print("COMPARAÇÃO GERAL DE MODELOS")
print("=" * 70)
print(df_resultados_geral.to_string(index=False))

print("\n" + "=" * 70)
print("✓ AVALIAÇÃO CONCLUÍDA COM SUCESSO")
print("=" * 70)

## 10 · Conclusão

Análise completa de classificação de notícias reais vs falsas em português:
- **Baseline**: Linear SVC e Logistic Regression (com TF-IDF)
- **Transformer**: BERTimbau fine-tuned
- **Resultados**: Matriz de confusão e comparação de desempenho dos modelos